<a href="https://colab.research.google.com/github/andresjaimes2004/tesis-desercion/blob/main/notebooks/04b_robustez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04b · Análisis de robustez (correcciones del jurado)

Complementa el cuaderno 04. Usa el modelo final y los hiperparámetros guardados en `reports/optimizacion.csv`, sin volver a optimizar, y no modifica el conjunto de prueba. Responde a cinco observaciones del jurado:

1. Fuga temporal: estudiantes que ya no presentaban evaluaciones en el semestre de corte.
2. Sensibilidad a las variables Matrícula al día y Deudor.
3. Comparación estadística entre algoritmos (DeLong y bootstrap pareado).
4. Calibración: modelo sin ponderación y recalibraciones de Platt e isotónica.
5. Desempeño por subgrupos.

**Tiempo estimado en Colab: entre 5 y 15 minutos.** Requiere haber ejecutado antes el cuaderno 04.

In [1]:
# Preparar el entorno (igual que en el cuaderno 00)
import os, sys
REPO_URL = "https://github.com/andresjaimes2004/tesis-desercion.git"
REPO_DIR = "/content/tesis-desercion"
if "google.colab" in sys.modules and not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import os, json, warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from src import config, prep, evaluation as ev, robustez as rb
from src.data import load_raw, binarize_target
from src.utils import set_seed

set_seed()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)
df = prep.add_no_enrollment_flags(binarize_target(load_raw()))
idx_train, idx_test = prep.load_split()   # índices ordenados: reproduce los pliegues del cuaderno 04
y_test = df.loc[idx_test, config.BINARY_TARGET]

optimizacion = pd.read_csv(config.REPORTS / "optimizacion.csv")
mejores = {r.modelo: json.loads(r.mejores_parametros) for r in optimizacion.itertuples()}
modelo_final = "XGBoost"
parametros = mejores[modelo_final]
print("Modelo final:", modelo_final, parametros)

## 0. Reproducción del modelo final

Debe dar exactamente las métricas del cuaderno 04 (umbral 0,44, sensibilidad 0,845, precisión 0,779, AUC-ROC 0,935 al cierre del 2.º semestre). Incluye ahora el intervalo de confianza de la exactitud.

In [ ]:
momentos, salidas = ev.evaluate_moments(df, idx_train, idx_test, modelo_final, parametros)
s2 = salidas["semestre_2"]
ic = ev.bootstrap_ci(y_test, s2["p_test"], s2["umbral"])
print({k: (round(a, 3), round(b, 3)) for k, (a, b) in ic.items()})
momentos.round(4)

## 1. Fuga temporal

La categoría «Dropout» se asigna al final de la duración normal del programa. Se cuentan los estudiantes que inscribieron unidades sin presentar ninguna evaluación y se evalúa el modelo sin ellos: con el modelo original y reentrenándolo sin esos casos (umbral elegido de nuevo con validación cruzada).

In [ ]:
retirados = rb.withdrawn_table(df)
retirados.to_csv(config.REPORTS / "fuga_retirados.csv", index=False, encoding="utf-8-sig")
retirados.round(1)

In [ ]:
sin_retirados = rb.evaluate_without_withdrawn(df, idx_train, idx_test, modelo_final, parametros)
sin_retirados.to_csv(config.REPORTS / "fuga_evaluacion_sin_retirados.csv", index=False, encoding="utf-8-sig")
sin_retirados.round(4)

## 2. Sin Matrícula al día ni Deudor

La documentación del conjunto no precisa en qué momento se registran estas variables. Se repite el modelo de cada momento sin ellas.

In [ ]:
for m in ["matricula", "semestre_1", "semestre_2"]:
    r = rb.without_financial(df, idx_train, idx_test, modelo_final, parametros, moment=m)
    r.to_csv(config.REPORTS / f"sin_financieras_{m}.csv", index=False, encoding="utf-8-sig")
    display(r.round(4))

## 3. Comparación estadística entre algoritmos

Prueba de DeLong para curvas ROC correlacionadas (algoritmo rápido de Sun y Xu, 2014) y bootstrap pareado de 1.000 remuestreos sobre el conjunto de prueba, al cierre del 2.º semestre.

In [ ]:
comparacion, probabilidades = rb.compare_models(df, idx_train, idx_test, mejores)
comparacion.to_csv(config.REPORTS / "comparacion_delong.csv", index=False, encoding="utf-8-sig")
print("Con Bonferroni (3 comparaciones), el nivel de significación es 0,05 / 3 =", round(0.05 / 3, 4))
comparacion.round(4)

## 4. Calibración

Modelo final (con ponderación) frente al mismo XGBoost sin ponderación (`scale_pos_weight = 1`) y a dos recalibraciones del modelo ponderado con validación cruzada (Platt e isotónica). Cada variante elige su umbral con predicciones fuera de pliegue.

In [ ]:
calibracion, prob_calibracion = rb.calibration_alternatives(df, idx_train, idx_test, modelo_final, parametros)
calibracion.to_csv(config.REPORTS / "calibracion_alternativas.csv", index=False, encoding="utf-8-sig")
calibracion.round(4)

## 5. Desempeño por subgrupos

Sensibilidad y precisión en la prueba según género, edad al ingresar y jornada, con el umbral del modelo final.

In [ ]:
subgrupos = rb.subgroup_performance(df, idx_test, s2["p_test"], s2["umbral"])
subgrupos.to_csv(config.REPORTS / "desempeno_subgrupos.csv", index=False, encoding="utf-8-sig")
subgrupos.round(3)

Los resultados quedan en la carpeta `reports/` y corresponden al apartado «Análisis de robustez» del documento (Tablas 11 y 12).